# Gabarito do Dever de Casa — Regularização e Fine-Tuning Parcial em CIFAR-10
**ENG4502 — Introdução à Ciência de Dados · PUC-Rio**

Este documento contém a resolução oficial do dever de casa prático proposto sobre Transfer Learning.

## 0. Imports e Dispositivo

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Subset
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando o dispositivo: {device}')

Usando o dispositivo: cpu


## 1. Carregamento dos Dados com Data Augmentation

Aqui aplicamos as transformações estocásticas no pipeline de treino e redimensionamento padrão.

In [2]:
# ==========================================
# EXERCÍCIO 1 (Solução): pipelines com data augmentation
# ==========================================
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

train_full = datasets.CIFAR10(root='data', train=True, download=True, transform=data_transforms['train'])
val_full = datasets.CIFAR10(root='data', train=False, download=True, transform=data_transforms['val'])

def extract_balanced_subset(dataset, n_total):
    n_per_class = n_total // 10
    indices = []
    class_counts = {c: 0 for c in range(10)}
    for idx, (_, label) in enumerate(dataset):
        if class_counts[label] < n_per_class:
            indices.append(idx)
            class_counts[label] += 1
        if len(indices) == n_total:
            break
    return Subset(dataset, indices)

train_dataset = extract_balanced_subset(train_full, 5000)
val_dataset = extract_balanced_subset(val_full, 1000)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f'Treino: {len(train_dataset)} imagens | Validação: {len(val_dataset)} imagens')

Files already downloaded and verified
Files already downloaded and verified
Treino: 5000 imagens | Validação: 1000 imagens


## Parte A — Treinamento do Zero (Scratch) com Augmentation

In [3]:
# ==========================================
# EXERCÍCIO 2 (Solução): modelo do zero
# ==========================================
model_scratch = models.resnet18(weights=None)
model_scratch.fc = nn.Linear(model_scratch.fc.in_features, 10)
model_scratch = model_scratch.to(device)

In [4]:
def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(dataloader.dataset)

def evaluate(model, dataloader):
    model.eval()
    corrects = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)
    return corrects.double().item() / len(dataloader.dataset)

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer_scratch = optim.SGD(model_scratch.parameters(), lr=0.01, momentum=0.9)

for epoch in range(1, 6):
    t0 = time.time()
    loss = train_epoch(model_scratch, train_loader, criterion, optimizer_scratch)
    acc = evaluate(model_scratch, val_loader)
    print(f'Época {epoch}/5 | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | Tempo: {time.time()-t0:.1f}s')

Época 1/5 | Loss: 2.1150 | Val Acc: 29.40% | Tempo: 310.2s
Época 2/5 | Loss: 1.8340 | Val Acc: 34.20% | Tempo: 307.8s
Época 3/5 | Loss: 1.6890 | Val Acc: 39.80% | Tempo: 311.5s
Época 4/5 | Loss: 1.5430 | Val Acc: 42.50% | Tempo: 309.2s
Época 5/5 | Loss: 1.4120 | Val Acc: 44.30% | Tempo: 308.7s


## Parte B — Fine-Tuning Parcial (layer4 + fc)

Nesta etapa, descongelamos apenas as últimas representações convolucionais (`layer4`) e o classificador.

In [6]:
# ==========================================
# EXERCÍCIO 3 (Solução): Fine-Tuning parcial seletivo
# ==========================================
model_partial = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 1. Congela tudo
for param in model_partial.parameters():
    param.requires_grad = False

# 2. Descongelar layer4 convolucional
for param in model_partial.layer4.parameters():
    param.requires_grad = True

# 3. Modificarfc classificadora (que automaticamente requer gradientes)
num_features = model_partial.fc.in_features
model_partial.fc = nn.Linear(num_features, 10)
model_partial = model_partial.to(device)

total_params = sum(p.numel() for p in model_partial.parameters())
trainable_params = sum(p.numel() for p in model_partial.parameters() if p.requires_grad)
print(f'Total de Parâmetros: {total_params:,}')
print(f'Parâmetros Treináveis: {trainable_params:,} (deve ser aproximadamente 8.39M)')

Total de Parâmetros: 11,181,642
Parâmetros Treináveis: 8,398,858 (deve ser aproximadamente 8.39M)


### Taxas de Aprendizado Discriminativas

In [7]:
# ==========================================
# EXERCÍCIO 4 (Solução): Otimizador discriminativo
# ==========================================
optimizer_partial = optim.SGD([
    {'params': model_partial.layer4.parameters(), 'lr': 1e-4},
    {'params': model_partial.fc.parameters(), 'lr': 1e-3}
], momentum=0.9)

for epoch in range(1, 6):
    t0 = time.time()
    loss = train_epoch(model_partial, train_loader, criterion, optimizer_partial)
    acc = evaluate(model_partial, val_loader)
    print(f'Época {epoch}/5 | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | Tempo: {time.time()-t0:.1f}s')

Época 1/5 | Loss: 1.4230 | Val Acc: 78.40% | Tempo: 201.2s
Época 2/5 | Loss: 0.7410 | Val Acc: 81.20% | Tempo: 198.5s
Época 3/5 | Loss: 0.5120 | Val Acc: 83.30% | Tempo: 199.1s
Época 4/5 | Loss: 0.3950 | Val Acc: 84.10% | Tempo: 197.8s
Época 5/5 | Loss: 0.3120 | Val Acc: 84.80% | Tempo: 199.4s


## Questões para Discussão

1. **Comparando o Scratch com e sem regularização:** O uso de Data Augmentation melhorou a acurácia de validação do modelo treinado do zero ao final da Época 5 em relação ao benchmark de aula (37.8%)? Explique o porquê.
   *Resposta:* Sim, a acurácia subiu de 37.8% para 44.30%. O uso de inversões e rotações stocásticas (Data Augmentation) impede a rede de focar em padrões espaciais rígidos específicos dos poucos dados de treino (5.000 imagens). Isso força o modelo a aprender características invariantes de rotação e translação, aumentando seu poder de generalização e retardando o overfitting.

2. **Tradeoff do Fine-Tuning Parcial:** Em relação à acurácia e ao tempo observados em aula, o Fine-Tuning Parcial conseguiu aproximar-se do Fine-Tuning Completo? Explique por que treinar apenas a `layer4` (2.6M de parâmetros) é mais eficiente e seguro em datasets menores.
   *Resposta:* O Fine-Tuning Parcial atingiu 84.80% (muito acima dos 74.5% do Feature Extraction e aproximando-se dos 87.9% do Fine-Tuning Completo). Em termos de tempo de execução no CPU, ele levou ~200s por época (total de ~16.5 min), em comparação com os ~315s por época (~26.4 min) do Fine-Tuning Completo. Treinar apenas a `layer4` é mais seguro contra overfitting e muito mais eficiente porque limita a atualização a apenas 2.6M de pesos profundos (onde as características são de alto nível e específicas do domínio), mantendo congelados os 8.6M de pesos das primeiras camadas convolucionais que extraem features genéricas.